# 编码器-解码器架构
:label:`sec_encoder-decoder`

>[编码器与解码器的简易架构原理](https://www.bilibili.com/video/BV1E2SuB2EhJ)

正如我们在 :numref:`sec_machine_translation`中所讨论的，
机器翻译是序列转换模型的一个核心问题，
其输入和输出都是**长度可变的序列**。
为了处理这种类型的输入和输出，
我们可以设计一个包含两个主要组件的架构：
1. 第一个组件是一个*编码器*（encoder）：
它接受一个长度可变的序列作为输入，
并将其转换为具有固定形状的编码状态。
2. 第二个组件是*解码器*（decoder）：
它将固定形状的编码状态映射到长度可变的序列。

这被称为*编码器-解码器*（encoder-decoder）架构，
如 :numref:`fig_encoder_decoder` 所示。

![编码器-解码器架构](../img/encoder-decoder.svg)
:label:`fig_encoder_decoder`

我们以英语到法语的机器翻译为例：
给定一个英文的输入序列：“They”“are”“watching”“.”。
首先，这种“编码器－解码器”架构将长度可变的输入序列编码成一个“状态”，
然后对该状态进行解码，
一个词元接着一个词元地生成翻译后的序列作为输出：
“Ils”“regordent”“.”。
由于“编码器－解码器”架构是形成后续章节中不同序列转换模型的基础，
因此本节将把这个架构转换为接口方便后面的代码实现。



## (**编码器**)

在编码器接口中，我们只指定**长度可变的序列**作为编码器的输入`X`。
任何继承这个`Encoder`基类的模型将完成代码实现。


In [2]:
from torch import nn


#@save
class Encoder(nn.Module):
    """编码器-解码器架构的基本编码器接口"""
    def __init__(self, **kwargs):
        super(Encoder, self).__init__(**kwargs)

    def forward(self, X, *args):
        raise NotImplementedError

## [**解码器**]

在下面的解码器接口中，我们新增一个`init_state`函数，
用于将编码器的输出（`enc_outputs`）转换为编码后的状态。
注意，<u>此步骤可能需要额外的输入</u>，例如：输入序列的有效长度，
这在 :numref:`subsec_mt_data_loading`中进行了解释。
为了逐个地生成*长度可变*的词元序列，
解码器在每个时间步都会将*输入*
（例如：在前一时间步生成的词元）和*编码后的状态*
映射成当前时间步的*输出词元*。

- 输入：这是解码器自己的历史输出
- 编码后的状态：这是编码器对整个源序列（如英文句子）
例如：
编码器读完 "Hello, how are you?" 后，将其压缩成一个“语义摘要” 。
解码器在生成每个法语词时，都参考这个摘要。
这句话描述的是**序列到序列**（Seq2Seq）的核心工作机制。我们来逐部分拆解，用通俗语言 + 技术细节解释：

- 解码器内部（RNN/Transformer）将上述两个输入**融合**，通过神经网络计算：
  $$
  \text{output}_t = \text{Decoder}(\text{previous\_token}, \text{encoded\_state})
  $$
    - 输出是一个**概率分布** over 词表（vocabulary）
    - 通过采样或贪心选择（取概率最大者）得到具体词元

> ✅ 例如：  
> 输入：前一词 = "Bonjour"，编码状态 = 对 "Hello" 的理解  
> 输出：`P("le")=0.1, P("vous")=0.8, P("<eos>")=0.1` → 选择 "vous"

📊 技术实现对比（不同模型）

| 模型 | “编码后的状态”是什么？ | “输入”如何处理？ |
|------|----------------------|----------------|
| **经典 Seq2Seq **(RNN) | 单个上下文向量 $c$（编码器最后隐藏态） | 前一词的 embedding + $c$ 拼接后输入 RNN |
| **带 Attention 的 RNN** | 所有源词的隐藏态 $[h_1, h_2, ..., h_T]$ | 前一词 embedding + **动态加权的上下文向量** |
| **Transformer 解码器** | 编码器输出的 key/value 矩阵 | 前面所有已生成词（通过 masked self-attention） + 编码器状态（cross-attention） |

> 💡 无论哪种模型，**核心思想不变**：  
> **当前输出 = f(自己刚说的话, 对原文的理解)**

---

❓ 常见疑问解答

### Q1: 为什么不能一次性生成整个句子？
- 因为语言具有**顺序依赖性**（“I love” → 下一个词大概率是名词）
- 自回归生成能保证**局部流畅性**

### Q2: 训练时和推理时输入一样吗？
- **训练时**：使用**真实目标词**作为下一时间步输入（teacher forcing）
- **推理时**：使用**自己预测的词**作为下一时间步输入

> ✅ 例如训练时：即使模型在 $t=1$ 预测错成 "Salut"，$t=2$ 的输入仍是真实词 "Bonjour"

---

✅ 总结

| 成分 | 是什么 | 作用 |
|------|------|------|
| **输入**（前一词元） | 解码器自己上一步的输出 | 保证生成序列的**内部连贯性** |
| **编码后的状态** | 编码器对源序列的表示 | 提供**源语言语义信息**，指导翻译 |
| **映射** | 解码器神经网络（RNN/Transformer） | 融合两者，预测下一个最可能的词 |

> 🌟 **本质**：解码器是一个**条件语言模型** ——  
> 它学习的是 $P(y_t \mid y_{<t}, x)$，即“**在已知原文 $x$ 和已生成部分 $y_{<t}$ 的条件下，预测下一个词 $y_t$**”。


In [3]:
#@save
class Decoder(nn.Module):
    """编码器-解码器架构的基本解码器接口"""
    def __init__(self, **kwargs):
        super(Decoder, self).__init__(**kwargs)

    def init_state(self, enc_outputs, *args):
        raise NotImplementedError

    def forward(self, X, state):
        raise NotImplementedError

## [**合并编码器和解码器**]

总而言之，“编码器-解码器”架构包含了一个编码器和一个解码器，
并且还拥有可选的额外的参数。
在前向传播中，编码器的输出用于生成编码状态，
这个状态又被解码器作为其输入的一部分。


In [4]:
#@save
class EncoderDecoder(nn.Module):
    """编码器-解码器架构的基类"""
    def __init__(self, encoder, decoder, **kwargs):
        super(EncoderDecoder, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X, *args):
        enc_outputs = self.encoder(enc_X, *args)
        dec_state = self.decoder.init_state(enc_outputs, *args)
        return self.decoder(dec_X, dec_state)

“编码器－解码器”体系架构中的术语*状态*
会启发人们使用具有状态的神经网络来实现该架构。
在下一节中，我们将学习如何应用循环神经网络，
来设计基于“编码器－解码器”架构的序列转换模型。

## 小结

* “编码器－解码器”架构可以将长度可变的序列作为输入和输出，因此适用于机器翻译等序列转换问题。
* 编码器将长度可变的序列作为输入，并将其转换为具有固定形状的编码状态。
* 解码器将具有固定形状的编码状态映射为长度可变的序列。

## 练习

1. 假设我们使用神经网络来实现“编码器－解码器”架构，那么编码器和解码器必须是同一类型的神经网络吗？
1. 除了机器翻译，还有其它可以适用于”编码器－解码器“架构的应用吗？


[Discussions](https://discuss.d2l.ai/t/2779)
